# P8: Redes neuronales para habla

**Contenido del notebook.**
1. Construir un vector de entrada a partir de los MFCC de un audio.
2. Entrenar y evaluar un MLP para clasificar dígitos aislados.

Al igual que en P6, empleamos el *Free Spoken Digit Dataset* (FSDD) y reutilizamos su carga de datos, `etiqueta_digito` y  `extraer_mfcc`. Ejecutar desde la misma carpeta que P6 para usar los audios ya descargados.

In [1]:
import librosa
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from copy import deepcopy
from pathlib import Path
from IPython.display import Audio, display
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score, ConfusionMatrixDisplay

np.random.seed(42)
torch.manual_seed(42)
torch.set_num_threads(2)
np.set_printoptions(precision=3, suppress=True)

## Ejercicio 3. Obtención del vector de entrada a partir de un audio

Cada archivo contiene un dígito completo. Las etiquetas son los dígitos de 0 a 9. La partición está fijada por hablante: `jackson`, `nicolas`, `theo` y `yweweler` para entrenamiento (2000 audios), `george` para validación (500) y `lucas` para test (500).

- Ejecutar la carga, la partición y la extracción MFCC provistas. Escuchar el ejemplo y observar su matriz de MFCC.
- Completar `resumir_audio` para obtener la media y el desvío temporal de cada coeficiente, y concatenarlos en un vector de dimensión 26, según la definición que aparece más abajo.
- Ejecutar la normalización provista y responder las preguntas de interpretación al final del ejercicio.

In [2]:
# Ejecutar una sola vez. Si ya existe la carpeta, no vuelve a descargar
data_dir = Path('free-spoken-digit-dataset/recordings')
if not data_dir.exists():
    print('No se encontró FSDD. Intentando clonar el repositorio...')
    import subprocess, sys
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/Jakobovski/free-spoken-digit-dataset.git'
    ], check=True)
else:
    print('FSDD encontrado.')

wav_paths = sorted(data_dir.glob('*.wav'))
print(f'Cantidad de audios encontrados: {len(wav_paths)}')
print('Ejemplo:', wav_paths[0].name if wav_paths else 'sin archivos')

def etiqueta_digito(path):
    # Formato FSDD: digito_nombre_indice.wav, por ejemplo 3_jackson_12.wav
    return int(path.stem.split('_')[0])

etiquetas_posibles = np.array([etiqueta_digito(p) for p in wav_paths])
print('Dígitos presentes:', sorted(set(etiquetas_posibles.tolist())))

FSDD encontrado.
Cantidad de audios encontrados: 3000
Ejemplo: 0_george_0.wav
Dígitos presentes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


In [3]:
hablantes_train = ['jackson', 'nicolas', 'theo', 'yweweler']
hablantes_val = ['george']
hablantes_test = ['lucas']

paths_train = [p for p in wav_paths if p.stem.split('_')[1] in hablantes_train]
paths_val = [p for p in wav_paths if p.stem.split('_')[1] in hablantes_val]
paths_test = [p for p in wav_paths if p.stem.split('_')[1] in hablantes_test]

etiquetas_train = np.array([etiqueta_digito(p) for p in paths_train])
etiquetas_val = np.array([etiqueta_digito(p) for p in paths_val])
etiquetas_test = np.array([etiqueta_digito(p) for p in paths_test])

print(f'Train: {len(paths_train)} audios\nValidación: {len(paths_val)} audios\nTest: {len(paths_test)} audios')

Train: 2000 audios
Validación: 500 audios
Test: 500 audios


### Extracción de MFCC

La función y su configuración son las mismas de P6 y recortamos silencios con `top_db=25`. Cada matriz tiene forma $T_i\times13$: una fila por frame y una columna por coeficiente.

In [ ]:
sr = 8000
n_mfcc = 13
hop = int(0.010 * sr) # 10 ms
win = int(0.025 * sr) # 25 ms
n_fft = 256

def extraer_mfcc(path):
    audio, sr_audio = librosa.load(path, sr=sr)
    audio, _ = librosa.effects.trim(audio, top_db=25)
    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=sr_audio,
        n_mfcc=n_mfcc,
        n_fft=n_fft,
        hop_length=hop,
        win_length=win,
        n_mels=26,
    ).T
    return mfcc.astype(float)

Y_train = [extraer_mfcc(p) for p in paths_train]
Y_val = [extraer_mfcc(p) for p in paths_val]
Y_test = [extraer_mfcc(p) for p in paths_test]

print('Ejemplo de secuencia MFCC:', Y_train[0].shape)

C:\Users\Dante P\miniconda3\envs\habla\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
idx = 0
display(Audio(filename=str(paths_test[idx])))
lim = np.max(np.abs(Y_test[idx]))
fig, ax = plt.subplots(figsize=(8, 3))
im = ax.imshow(Y_test[idx].T, origin='lower', aspect='auto', cmap='RdBu_r', vmin=-lim, vmax=lim)
ax.set_xlabel('Frame $t$')
ax.set_ylabel('Coeficiente $d$')
ax.set_title(f'MFCC de {paths_test[idx].name}')
fig.colorbar(im, ax=ax, label=r'$c_d[t]$')
plt.tight_layout()
plt.show()

### Resumen y normalización

Para el audio $i$, con $T_i$ frames y vectores $\mathbf y_t=[c_0[t],\ldots,c_{12}[t]]^T$, calcular

$$
\overline{\mathbf y}=\frac{1}{T_i}\sum_{t=1}^{T_i}\mathbf y_t,
\qquad
\boldsymbol\sigma=\sqrt{\frac{1}{T_i}\sum_{t=1}^{T_i}(\mathbf y_t-\overline{\mathbf y})^{\odot2}},
\qquad
\mathbf u=\begin{bmatrix}\overline{\mathbf y}\\\boldsymbol\sigma\end{bmatrix}\in\mathbb R^{26}.
$$

El cuadrado y la raíz se aplican componente a componente. Usar `ddof=0` para el desvío, es decir, asumir que el divisor es solamente $T_i$. Las primeras 13 entradas de $\mathbf u$ deben ser las medias y las últimas 13 los desvíos. Luego normalizamos estas 26 features con `StandardScaler`, el cual es ajustado sólo en train.

In [ ]:
def resumir_audio(Y_audio):
    media = np.mean(Y_audio, axis=0)  ### <-- COMPLETAR
    desvio = np.std(Y_audio, axis=0, ddof=0) ### <-- COMPLETAR
    return np.concatenate([media, desvio])

U_train = np.vstack([resumir_audio(y) for y in Y_train])
U_val = np.vstack([resumir_audio(y) for y in Y_val])
U_test = np.vstack([resumir_audio(y) for y in Y_test])

print('Entradas del MLP:', U_train.shape)
print('Mismo resumen al invertir frames:', np.allclose(resumir_audio(Y_test[idx]), resumir_audio(Y_test[idx][::-1])))
print("Esto sirve para verificar que hiciste bien los calculos, como vos promediaste en frames, debe dar igual si lo invertís o no")

In [ ]:
# Normalización global: se ajusta en train y se aplica a train/val/test
scaler = StandardScaler()
scaler.fit(U_train)
U_train = scaler.transform(U_train)
U_val = scaler.transform(U_val)
U_test = scaler.transform(U_test)

# Convertimos a tensores para poder usarlos en PyTorch
U_train = torch.tensor(U_train, dtype=torch.float32)
U_val = torch.tensor(U_val, dtype=torch.float32)
U_test = torch.tensor(U_test, dtype=torch.float32)
labels_train = torch.tensor(etiquetas_train, dtype=torch.long)
labels_val = torch.tensor(etiquetas_val, dtype=torch.long)

# Creamos el dataset y el dataloader
dataset_train = TensorDataset(U_train, labels_train)
loader = DataLoader(dataset_train, batch_size=32, shuffle=True, generator=torch.Generator().manual_seed(42))
print("DataLoader maneja el procesamiento de datos para cada set y epoch, el shuffle dice que en orden en el que mostar cada data point es random, los paquetes de 32 tiene distitos orden")
print("")

**Responder:** ¿Qué representan las primeras y las últimas 13 entradas? ¿Qué información pierde el resumen?
¿Por qué el scaler se ajusta sólo en train?

## Ejercicio 4. MLP: entrenamiento e inferencia

Usamos una red $26\to32\to10$, con ReLU, batches de 32, 100 *epochs*, entropía cruzada media y Adam con tasa de aprendizaje $\eta_{\text{NN}}=10^{-3}$. El código conserva el *epoch* con menor pérdida en el set de validación y después evalúa en test.

- Completar las dos capas y el `forward` de `MLP`. Anticipar la cantidad de parámetros entrenables y contrastarla con la salida del código.
- Completar `criterio` y la pérdida para cada batch. `CrossEntropyLoss` recibe logits y etiquetas enteras. No aplicar softmax antes de la pérdida.
- Completar las tres líneas del entrenamiento que reinician los gradientes, calculan las derivadas y actualizan los parámetros. Graficar las curvas de aprendizaje.
- Completar la predicción de los dígitos de test mediante `argmax`. Ejecutar la evaluación, la matriz de confusión y el ejemplo con sus probabilidades.

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(26, 32) ### <-- COMPLETAR
        self.fc2 = nn.Linear(32, 10) ### <-- COMPLETAR

    def forward(self, u):
        h = torch.relu(self.fc1(u))  ### <-- COMPLETAR
        return self.fc2(h)

torch.manual_seed(42)
modelo = MLP()
print(modelo)
print('Parámetros entrenables:', sum(p.numel() for p in modelo.parameters()))

In [ ]:
criterio = nn.CrossEntropyLoss() ### <-- COMPLETAR
#Otras opciones: nn.MSE()
optimizador = torch.optim.Adam(modelo.parameters(), lr=1e-3, weight_decay=0.01)


def entrenar_epoch():
    #Acordate que en una epoch vemos TODO el train_dataset
    modelo.train()
    total_loss, correctos = 0.0, 0
    for u_batch, labels_batch in loader: #esto te escupe los 32 elementos 
        logits = modelo(u_batch)
        loss = criterio(logits, labels_batch) ### <-- COMPLETAR
        optimizador.zero_grad() ### <-- COMPLETAR (resetear gradientes)
        loss.backward() ### <-- COMPLETAR (hacer backpropagation)
        optimizador.step() ### <-- COMPLETAR (actualizar parámetros)
        total_loss += loss.item() * len(labels_batch)
        correctos += (logits.argmax(dim=1) == labels_batch).sum().item()
    return total_loss / len(dataset_train), correctos / len(dataset_train)

def evaluar(u, labels):
    modelo.eval()
    with torch.no_grad():
        logits = modelo(u)
        loss = criterio(logits, labels).item()
        acc = (logits.argmax(dim=1) == labels).float().mean().item()
    return loss, acc

**Regla de oro:**
* Borro gradientes
* Hacer backpropagarion: solo se calcula gradientes
* Actualizar parámetros
Con esto minimizás la loss **del batch**

Detalles de Adam:
* scheduler: Hacés que vayan decayendo el learning rate según una regla. Es obligatorio cuando usás otros optimizadores
* weight_decay: Es un tipo de regularizador, es decir, estrategías que te permiten reducir el overfitting
  * En particular, esto baja el valor medio.
  * Ojo! **NO** es una regularización L2!. 
  * Si ves la implementación, la adaptatividad del learning rate (de adam) está mezclado con el weight decay
* 3 valores típicos de learning rate: $1\text{e}-3$; $1\text{e}-5$ (para LLMS) y $3\text{e}-4$ (valor de Karpathy, demuestra si no sabés donde inicializar, conviene este valor)

In [ ]:
historial = []
mejor_loss = np.inf
for epoch in range(1, 101):
    loss_train, acc_train = entrenar_epoch()
    loss_val, acc_val = evaluar(U_val, labels_val)
    historial.append([loss_train, loss_val, acc_train, acc_val])
    if loss_val < mejor_loss:
        mejor_loss = loss_val
        mejor_epoch = epoch
        mejor_estado = deepcopy(modelo.state_dict())
    if epoch == 1 or epoch % 20 == 0:
        print(f'Epoch {epoch}: loss train={loss_train:.3f}, loss val={loss_val:.3f}, acc train={acc_train:.3f}, acc val={acc_val:.3f}')

modelo.load_state_dict(mejor_estado)
print('Epoch restaurado:', mejor_epoch)

In [ ]:
historial = np.array(historial)
epochs = np.arange(1, 101)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(epochs, historial[:, 0], label='Train')
axes[0].plot(epochs, historial[:, 1], label='Validación')
axes[0].set_ylabel('Entropía cruzada media (nats)')
axes[1].plot(epochs, historial[:, 2], label='Train')
axes[1].plot(epochs, historial[:, 3], label='Validación')
axes[1].set_ylabel('Accuracy')
for ax in axes:
    ax.set_xlabel('Epoch')
    ax.axvline(mejor_epoch, color='gray', linestyle=':')
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
modelo.eval()
with torch.no_grad():
    logits_test = modelo(U_test)
    y_pred = logits_test.argmax(dim=1).numpy() ### <-- COMPLETAR. Lo pasamos a numpy

acc = accuracy_score(etiquetas_test, y_pred)
print(f'Accuracy: {acc:.3f}')

cm = confusion_matrix(etiquetas_test, y_pred, labels=list(range(10)))
disp = ConfusionMatrixDisplay(cm, display_labels=list(range(10)))
disp.plot(cmap='Blues', values_format='d')
plt.title('Matriz de confusión para FSDD')
plt.tight_layout()
plt.show()

In [ ]:
idx = 200
with torch.no_grad():
    probabilidades = torch.softmax(modelo(U_test[idx:idx + 1]), dim=1)[0].numpy()
print('Dígito verdadero:', etiquetas_test[idx])
print('Dígito predicho: ', y_pred[idx])
display(Audio(filename=str(paths_test[idx])))

plt.figure(figsize=(7, 3))
plt.bar(np.arange(10), probabilidades)
plt.xlabel('Dígito')
plt.ylabel('Probabilidad a posteriori estimada')
plt.xticks(range(10))
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

**Interpretar.** Comparar la evolución de la pérdida y la accuracy en train y validación. ¿Por qué se recuperan los parámetros de un *epoch* anterior al máximo? Elegir una celda fuera de la diagonal de la matriz de confusión de test e indicar el dígito verdadero, el predicho y la cantidad de casos. ¿Qué nos permite evaluar el hecho de que los hablantes de train, validación y test sean distintos?

* Lo hacemos para ver que tan buena es la generalización. Si mezclas básicamente vas a estar likeando información (por el estilo de habla)